# Increment diagnostics — do the data actually need heavy-tailed + jump noise?

**Before** committing to α-stable innovations and a compound-Poisson jump term
(as in `AlphaStableLOB` / `JumpGateLOB`), we must show *from the data* that the
LOB increments are genuinely non-Gaussian and jumpy. Otherwise a simpler
near-Gaussian model is preferable and the modelling choice is unmotivated.

We run four standard diagnostics on the **best-level mid log-returns**
(`r = Δ log[(bid₀+ask₀)/2]`, computed within a contiguous session so we never
difference across a gap) for **every Nobitex crypto symbol** and for **Feishu
A-share** LOB data:

1. **Distribution shape** — histogram vs a fitted normal (log-y), Q–Q plot, and a
   log–log survival plot (a straight tail ⇒ power law). Plus a **Jarque–Bera**
   normality test.
2. **Hill tail index** `α̂` with a Hill plot across thresholds. **`α̂ < 2` ⇒
   infinite variance ⇒ supports an α-stable law**; `α̂ ≳ 4` ⇒ light-ish tails.
3. **Excess kurtosis** (Gaussian = 0). Strongly positive ⇒ fatter tails /
   outlier-prone. (Unstable if variance is truly infinite — reported as a
   descriptive sign only.)
4. **Formal jump tests** — **Barndorff-Nielsen–Shephard** (realized vs bipower
   variation, per session) and **Lee–Mykland** (per-return local-jump test).
   Significant jumps ⇒ a compound-Poisson component is empirically motivated.

**Verdict logic.** `α̂<2` or large excess kurtosis ⇒ heavy tails ⇒ α-stable;
significant BNS/LM jumps ⇒ jump component; neither ⇒ Gaussian is adequate and
the complex model *gains nothing* (the sanity limit).

## 1. Setup

In [ ]:
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import gamma as _gammafn

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda v: f"{v:.4f}")


def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists() and (p / "src").is_dir():
            return p
    raise RuntimeError(f"could not find Penny repo root above {start}")


REPO = _find_repo_root(Path.cwd())
os.chdir(REPO)
print("repo root:", REPO)

# --- what to diagnose ---------------------------------------------------------
CRYPTO_SYMBOLS = ["BTCIRT", "BNBIRT", "ETHIRT", "XRPIRT", "USDTIRT"]  # USDT = control
FEISHU_ASSETS = [  # 20 high-volume names (uniform + noisy) from the volume analysis
    "A002059", "A001750", "A000863", "A000369", "A001402",
    "A001472", "A001351", "A001438", "A000956", "A000613",
    "A002000", "A000295", "A001748", "A002042", "A000400",
    "A000875", "A000452", "A001208", "A002142", "A001663",
]
CRYPTO_DIR = REPO / "data" / "resampled" / "nobitex"
FEISHU_LOB = REPO / "data" / "stocks" / "feishu" / "lob_data_in_sample.parquet"

## 2. Diagnostic toolkit

Each estimator is standard; see the references in the header. All operate on a
1-D array of returns, and the jump tests operate **per session segment** (a
contiguous block with no gap) so bipower/realized variation are well defined.

In [ ]:
MU1 = np.sqrt(2.0 / np.pi)  # E|Z| for Z~N(0,1); bipower normaliser
MU43 = 2.0 ** (2.0 / 3.0) * _gammafn(7.0 / 6.0) / _gammafn(0.5)  # for tripower quarticity


def hill_estimator(x, ks=None):
    """Hill tail index α̂ of |x| across order statistics k (α̂<2 ⇒ infinite var).

    Returns (ks, alpha, se). α̂ = 1 / mean(log|x|_(i) − log|x|_(k+1)), i=1..k.
    """
    a = np.sort(np.abs(np.asarray(x, float)))[::-1]
    a = a[np.isfinite(a) & (a > 0)]
    n = len(a)
    if n < 50:
        return np.array([]), np.array([]), np.array([])
    if ks is None:
        ks = np.unique(np.linspace(10, int(0.15 * n), 80).astype(int))
        ks = ks[ks < n]
    la = np.log(a)
    alpha, se = [], []
    for k in ks:
        Hk = la[:k].mean() - la[k]
        al = 1.0 / Hk if Hk > 0 else np.nan
        alpha.append(al)
        se.append(al / np.sqrt(k) if np.isfinite(al) else np.nan)
    return np.asarray(ks), np.asarray(alpha), np.asarray(se)


def hill_at(x, frac=0.05):
    """Representative Hill α̂ (± asymptotic 95% CI) at the top `frac` of |x|."""
    ks, al, se = hill_estimator(x)
    if len(ks) == 0:
        return np.nan, np.nan, np.nan
    n = np.sum(np.isfinite(x) & (np.asarray(x, float) != 0))
    ktarget = max(10, int(frac * n))
    j = int(np.argmin(np.abs(ks - ktarget)))
    return float(al[j]), float(al[j] - 1.96 * se[j]), float(al[j] + 1.96 * se[j])


def bns_jump_test(r):
    """Barndorff-Nielsen–Shephard ratio jump test on one session's returns.

    Returns (z, RJ) with z ~ N(0,1) under 'no jumps'; z large positive ⇒ jump.
    RJ = (RV−BV)/RV is the relative jump contribution to variance.
    """
    r = np.asarray(r, float)
    r = r[np.isfinite(r)]
    n = len(r)
    if n < 10:
        return np.nan, np.nan
    RV = np.sum(r ** 2)
    BV = (1.0 / MU1 ** 2) * np.sum(np.abs(r[1:]) * np.abs(r[:-1]))
    if RV <= 0 or BV <= 0:
        return np.nan, np.nan
    ar = np.abs(r)
    TP = n * (MU43 ** -3) * np.sum(
        ar[2:] ** (4 / 3) * ar[1:-1] ** (4 / 3) * ar[:-2] ** (4 / 3)
    )
    RJ = (RV - BV) / RV
    theta = (np.pi ** 2 / 4.0) + np.pi - 5.0
    denom = np.sqrt(theta * (1.0 / n) * max(1.0, TP / BV ** 2))
    return (RJ / denom if denom > 0 else np.nan), RJ


def lee_mykland(r, K=None, sig=0.01):
    """Lee–Mykland (2008) per-return jump test on one session.

    Returns (n_jumps, n_tested). A return is a jump if its local-vol-standardised
    magnitude exceeds the Gumbel critical value for the session length.
    """
    r = np.asarray(r, float)
    r = r[np.isfinite(r)]
    n = len(r)
    if K is None:
        K = int(np.clip(np.sqrt(n) * 3, 15, 270))
    if n < K + 5:
        return 0, 0
    ar = np.abs(r)
    bp = ar[1:] * ar[:-1]                       # |r_j||r_{j-1}|, index j=1..n-1
    csum = np.concatenate([[0.0], np.cumsum(bp)])
    L = np.full(n, np.nan)
    for i in range(K, n):
        sig2 = (csum[i] - csum[i - (K - 1)]) / (K - 2)   # local bipower vol²
        if sig2 > 0:
            L[i] = r[i] / (MU1 * np.sqrt(sig2))
    tested = np.isfinite(L)
    m = int(tested.sum())
    if m < 5:
        return 0, 0
    c = np.sqrt(2.0 * np.log(m))
    Cn = c - (np.log(np.pi) + np.log(np.log(m))) / (2.0 * c)
    Sn = 1.0 / c
    thresh = Cn - Sn * np.log(-np.log(1.0 - sig))   # Gumbel critical value
    return int(np.sum(np.abs(L[tested]) > thresh)), m


def diagnose(pooled, segments, name):
    """Run every diagnostic; return a summary dict + a per-verdict string."""
    x = np.asarray(pooled, float)
    x = x[np.isfinite(x)]
    n = len(x)
    ek = float(stats.kurtosis(x, fisher=True, bias=False)) if n > 3 else np.nan
    jb_p = float(stats.jarque_bera(x)[1]) if n > 7 else np.nan
    a, alo, ahi = hill_at(x)
    # jump tests aggregated over session segments
    z_flags, seg_n, lm_j, lm_t = 0, 0, 0, 0
    for seg in segments:
        z, _ = bns_jump_test(seg)
        if np.isfinite(z):
            seg_n += 1
            z_flags += int(z > 1.645)   # one-sided 5%
        j, t = lee_mykland(seg)
        lm_j += j
        lm_t += t
    bns_frac = z_flags / seg_n if seg_n else np.nan
    lm_frac = lm_j / lm_t if lm_t else np.nan

    heavy = (np.isfinite(a) and a < 4.0) or (np.isfinite(ek) and ek > 1.0)
    inf_var = np.isfinite(a) and a < 2.0
    jumpy = (np.isfinite(bns_frac) and bns_frac > 0.10) or (
        np.isfinite(lm_frac) and lm_frac > 0.002
    )
    if inf_var:
        tail_v = "α<2 → α-stable (infinite variance)"
    elif heavy:
        tail_v = "heavy-tailed (finite var) → Student-t / mild stable"
    else:
        tail_v = "≈ Gaussian tails"
    verdict = f"{tail_v}; " + ("JUMPS detected → jump component" if jumpy
                               else "no significant jumps")
    return {
        "name": name, "n": n, "excess_kurt": ek, "jb_p": jb_p,
        "hill_alpha": a, "hill_lo": alo, "hill_hi": ahi,
        "bns_jump_seg_%": 100 * bns_frac if np.isfinite(bns_frac) else np.nan,
        "lm_jump_%": 100 * lm_frac if np.isfinite(lm_frac) else np.nan,
        "verdict": verdict,
    }


print(f"toolkit ready — MU1={MU1:.4f}, MU43={MU43:.4f}")

## 3. Increment extraction

`r = Δ log[(bid₀+ask₀)/2]` within each contiguous session. **Crypto** sessions =
calendar days of the 10-second stream; **Feishu** sessions = (asset, trade day)
of the 5-min stream. Each extractor returns `(pooled, segments)` — the flat array
for distribution stats, and the list of per-session arrays for the jump tests.

In [ ]:
def crypto_increments(symbol):
    p = CRYPTO_DIR / f"{symbol}.parquet.gz"
    df = pd.read_parquet(p, columns=["timestamp_utc", "bids[0].price", "asks[0].price"])
    mid = (df["bids[0].price"] + df["asks[0].price"]).to_numpy(float) / 2.0
    day = pd.to_datetime(df["timestamp_utc"]).dt.normalize().to_numpy()
    ok = np.isfinite(mid) & (mid > 0)
    logm, day = np.log(mid[ok]), day[ok]
    segments, pooled = [], []
    for d in np.unique(day):
        s = logm[day == d]
        if len(s) > 2:
            r = np.diff(s)
            segments.append(r)
            pooled.append(r)
    return (np.concatenate(pooled) if pooled else np.array([])), segments


def feishu_increments(assets):
    import pyarrow.compute as pc
    import pyarrow.dataset as ds
    cols = ["asset_id", "trade_day_id", "time", "bid_price_1", "ask_price_1"]
    tbl = ds.dataset(FEISHU_LOB, format="parquet").to_table(
        columns=cols, filter=pc.field("asset_id").isin(assets)
    )
    df = tbl.to_pandas()
    df["mid"] = (df["bid_price_1"] + df["ask_price_1"]) / 2.0
    df = df[np.isfinite(df["mid"]) & (df["mid"] > 0)]
    df = df.sort_values(["asset_id", "trade_day_id", "time"])
    segments, pooled = [], []
    for _, g in df.groupby(["asset_id", "trade_day_id"], sort=False):
        s = np.log(g["mid"].to_numpy(float))
        if len(s) > 2:
            r = np.diff(s)
            segments.append(r)
            pooled.append(r)
    return (np.concatenate(pooled) if pooled else np.array([])), segments


# extract everything once
DATA = {}
for sym in CRYPTO_SYMBOLS:
    try:
        DATA[("crypto", sym)] = crypto_increments(sym)
        print(f"crypto {sym:<8} increments={len(DATA[('crypto', sym)][0]):>8,} "
              f"sessions={len(DATA[('crypto', sym)][1])}")
    except Exception as e:
        print(f"crypto {sym}: skipped ({e})")
try:
    DATA[("feishu", "A-shares")] = feishu_increments(FEISHU_ASSETS)
    print(f"feishu   {'panel':<8} increments={len(DATA[('feishu', 'A-shares')][0]):>8,} "
          f"sessions={len(DATA[('feishu', 'A-shares')][1])}")
except Exception as e:
    print(f"feishu: skipped ({e})")

## 4. Per-symbol distribution + tail plots

Four panels per series: histogram vs fitted normal (log-y — tails visible),
Q–Q vs normal (points bending off the line at the ends ⇒ heavy tails), log–log
survival of |r| (a straight descending tail ⇒ power law; slope ≈ −α), and the
Hill plot (α̂ vs k; a stable level **below the α=2 line** ⇒ infinite variance).

In [ ]:
def plot_series(pooled, name):
    x = np.asarray(pooled, float)
    x = x[np.isfinite(x)]
    if len(x) < 50:
        print(f"{name}: too few points"); return
    xs = x / (x.std() + 1e-12)                 # standardised for the normal overlay
    fig, ax = plt.subplots(1, 4, figsize=(18, 3.6))

    ax[0].hist(xs, bins=200, density=True, alpha=0.6, color="steelblue")
    g = np.linspace(xs.min(), xs.max(), 400)
    ax[0].plot(g, stats.norm.pdf(g), "r-", lw=1.2, label="N(0,1)")
    ax[0].set_yscale("log"); ax[0].set_title(f"{name}\nhist vs normal (log-y)")
    ax[0].legend(fontsize=7)

    stats.probplot(xs, dist="norm", plot=ax[1])
    ax[1].set_title("Q–Q vs normal")
    ax[1].get_lines()[0].set_markersize(2)

    a = np.sort(np.abs(x))[::-1]; a = a[a > 0]
    surv = np.arange(1, len(a) + 1) / len(a)
    ax[2].loglog(a, surv, ".", ms=2)
    ax[2].set_title("log–log survival of |r|\n(straight tail ⇒ power law)")
    ax[2].set_xlabel("|r|"); ax[2].set_ylabel("P(|R|>|r|)")

    ks, al, se = hill_estimator(x)
    if len(ks):
        ax[3].plot(ks, al, "-", color="darkgreen")
        ax[3].fill_between(ks, al - 1.96 * se, al + 1.96 * se, alpha=0.2, color="green")
        ax[3].axhline(2.0, color="red", ls="--", lw=1, label="α=2 (finite-var boundary)")
        ax[3].set_title("Hill plot  α̂ vs k"); ax[3].set_xlabel("k (top order stats)")
        ax[3].set_ylabel("α̂"); ax[3].set_ylim(0, min(8, np.nanmax(al) * 1.2))
        ax[3].legend(fontsize=7)
    plt.tight_layout(); plt.show()


for (dom, sym), (pooled, _segs) in DATA.items():
    plot_series(pooled, f"{dom}:{sym}")

## 5. Summary table + verdict

One row per series with every diagnostic and its verdict. Read the columns as:
`hill_alpha` < 2 (CI upper bound below 2 is strong) ⇒ α-stable; `excess_kurt`
≫ 0 and `jb_p` ≈ 0 ⇒ non-Gaussian; `bns_jump_seg_%` well above the nominal 5%
and `lm_jump_%` > 0 ⇒ real jumps.> **Caveat — jump tests on discrete / zero-inflated high-frequency data.** At> 10-second (crypto) frequency the mid is *zero-inflated*: it often does not move> between snapshots, so many consecutive returns are 0. This shrinks the bipower> variation (products of consecutive `|r|`) relative to realized variation and> makes **BNS structurally over-detect** — `bns_jump_seg_%` near 100% is partly> this artifact, not all genuine jumps. Trust the per-return **Lee–Mykland**> `lm_jump_%`, the **excess kurtosis**, and the **Hill α̂** as the robust> evidence; read BNS as corroborating, not decisive. (Sub-sampling to a coarser> grid, e.g. 1–5 min, or pre-filtering zero-returns, sharpens BNS if needed.)> **On the huge sample kurtosis.** A Hill `α̂ ≈ 2.5–3` implies a *finite variance*> but an **infinite fourth moment** (kurtosis needs `α>4`), so the sample excess> kurtosis of 10²–10³ is expected to be large and unstable — itself a signature> of very heavy tails, consistent with a near-stable / Student-t law.

In [ ]:
rows = [diagnose(pooled, segs, f"{dom}:{sym}")
        for (dom, sym), (pooled, segs) in DATA.items()]
summary = pd.DataFrame(rows).set_index("name")
cols = ["n", "excess_kurt", "jb_p", "hill_alpha", "hill_lo", "hill_hi",
        "bns_jump_seg_%", "lm_jump_%", "verdict"]
disp = summary[cols].copy()
sty = (disp.style
       .format({"n": "{:,.0f}", "excess_kurt": "{:.1f}", "jb_p": "{:.1e}",
                "hill_alpha": "{:.2f}", "hill_lo": "{:.2f}", "hill_hi": "{:.2f}",
                "bns_jump_seg_%": "{:.1f}", "lm_jump_%": "{:.3f}"})
       .background_gradient(cmap="OrRd", subset=["excess_kurt"])
       .background_gradient(cmap="YlGn_r", subset=["hill_alpha"]))
display(sty)

print("\nverdicts:")
for name, r in summary.iterrows():
    print(f"  {name:<18} {r['verdict']}")

### What to write in the paper

Fill the sentence from the table above, e.g.:

> *Across all Nobitex crypto symbols the mid-return increments have excess
> kurtosis of order 10²–10³ and Jarque–Bera p ≈ 0 (normality decisively
> rejected); the Hill tail index estimates sit [below / near] 2, and the
> BNS and Lee–Mykland tests flag jumps in a large fraction of sessions. These
> diagnostics support modelling the increments with heavy-tailed α-stable
> innovations and a compound-Poisson jump component. The stablecoin control
> (USDTIRT) is comparatively light-tailed, confirming the diagnostics
> discriminate.*

If instead a series comes out near-Gaussian with no jumps, the honest
conclusion is that the simpler model suffices for it — the sanity limit.